Client Intelligence - Energical Decision Platform

Analyse RFM (Recency, Frequency, Monetary) pour segmenter les clients :
Champions, clients a risque, clients perdus.

 1. Import des librairies

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

2. Connexion a la base de donnees

In [2]:
load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

3. Chargement des donnees clients

Recuperation de la table customers, qui contient deja des indicateurs 
agreges utiles pour le RFM (dates, nombre de commandes, montants).

In [3]:
query = """
SELECT customer_id_stage, customer_type_inferred, wilaya, 
       first_order_date, last_order_date, orders_count, 
       total_amount, average_basket
FROM customers
"""

df_customers = pd.read_sql(query, engine)

df_customers.shape

(4946, 8)

 4. Calcul de la Recency (en jours)

Calcul du nombre de jours depuis la derniere commande de chaque client,
par rapport a la date la plus recente disponible dans les donnees.

In [4]:
df_customers["last_order_date"] = pd.to_datetime(df_customers["last_order_date"])

reference_date = df_customers["last_order_date"].max()

df_customers["recency_days"] = (
    reference_date - df_customers["last_order_date"]
).dt.days

df_customers[["customer_id_stage", "last_order_date", "recency_days"]].head()

,customer_id_stage,last_order_date,recency_days
0,CLT_S00001,2026-06-02,47
1,CLT_S00002,2026-07-13,6
2,CLT_S00003,2026-06-10,39
3,CLT_S00004,2026-05-24,56
4,CLT_S00005,2023-08-09,1075


 5. Frequency et Monetary

Frequency = orders_count (deja disponible)
Monetary = total_amount (deja disponible)

Ces deux indicateurs sont deja agreges dans la table customers.

In [ ]:
df_customers[["customer_id_stage", "orders_count", "total_amount"]].describe()

,orders_count,total_amount
count,4946.000000,4.946000e+03
mean,1.869187,6.592939e+04
std,20.617428,1.203012e+06
min,1.000000,0.000000e+00
25%,1.000000,4.511000e+03
50%,1.000000,1.558200e+04
75%,1.000000,3.549750e+04
max,1381.000000,8.076708e+07


: 

 Note sur les valeurs extremes (outlier)

Un client presente 1381 commandes, largement superieur au reste des clients 
(2eme valeur : 411). Ce client est probablement une entreprise B2B mal 
classee "B2C" par l'heuristique customer_type_inferred (a confirmer avec Soulef).

Il n'est pas exclu des donnees, mais son score sera naturellement au maximum 
dans le scoring RFM base sur les quantiles.